In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   KazPunct-Hackathon                                             ║
# ║   Target: ~0.85+ Macro F1  |  Runtime: ~25-35 min on T4 GPU      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, re, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler        # fp16 mixed precision
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

try:
    from datasets import load_dataset
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False

# ══════════════════════════════════════════════
# CONFIG  — every key decision is here
# ══════════════════════════════════════════════
CFG = dict(
    model_name      = "xlm-roberta-large",

    # ── Speed knobs ───────────────────────────
    max_len         = 128,      # ↓ was 256  → 2x faster, minimal F1 loss
    batch_size      = 32,       # ↑ was 16   → fewer steps per epoch
    max_corpus_rows = 40_000,   # ↓ was 80k  → still diverse, 3x faster load
    epochs          = 14,        # ↑ was 4    → compensates for less data

    # ── Optimizer ─────────────────────────────
    lr           = 2e-5,
    warmup_ratio = 0.06,
    weight_decay = 0.01,
    grad_clip    = 1.0,
    seed         = 42,

    # ── Class weights ─────────────────────────
    # QUESTION is only 0.3% of tokens — model ignores it without heavy weight
    class_weights = {
        "O":        0.05,
        "COMMA":    1.5,
        "PERIOD":   1.0,
        "QUESTION": 6.0,    # ↑ was 4.0 — key change for QUESTION F1
    },

    # ── Paths ─────────────────────────────────
    train_path  = "/kaggle/input/competitions/kaz-punct-hackathon/train_example.csv",
    test_path   = "/kaggle/input/competitions/kaz-punct-hackathon/test.csv",
    output_path = "submission.csv",
)

LABEL2ID   = {"O": 0, "COMMA": 1, "PERIOD": 2, "QUESTION": 3}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 4
PUNCT_LABELS = ["COMMA", "PERIOD", "QUESTION"]
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP    = (DEVICE.type == "cuda")   # fp16 only on GPU

print(f"Device: {DEVICE}  |  Mixed precision (fp16): {USE_AMP}")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])

# ══════════════════════════════════════════════
# KAZAKH GRAMMAR RULES  (post-processing)
# ══════════════════════════════════════════════
#
# These are HARD overrides applied AFTER the model predicts.
# The model is already good at PERIOD; these rules fix QUESTION
# which is rare (0.3%) and often missed.
#
# Rule 1 — Сұрау демеулік шылаулары (Question modal particles)
#   Казақ тілінде сұрақ сөйлем белгісі:
#   ма / ме / па / пе / ба / бе
#   Бұл сөздер сөйлем соңында тұрса → міндетті QUESTION
#   Мысалы: "белгіленді ме" → "ме" = сұрау демеулігі → QUESTION
#
# Rule 2 — Соңғы токен PERIOD болуы керек
#   Әр жол 2-3 сөйлемнен тұрады. Соңғы сөйлем міндетті нүктемен
#   аяқталуы керек. Егер модель соңғы сөзге O берсе → PERIOD.
#
# Rule 3 — Сұраулы есімдіктер (Question pronouns/adverbs)
#   кім, не, қайда, қайдан, қашан, қалай, неше, қанша, неліктен
#   Бұлар бар жолда QUESTION болу ықтималдығы жоғары.
#   (Soft rule — тек модель O берген соңғы баяндауышта қолданылады)

# Exact question particles — these ALWAYS force QUESTION when at end of sentence
QUESTION_PARTICLES = {"ма", "ме", "па", "пе", "ба", "бе"}

# Question words — increase QUESTION probability but don't force
QUESTION_WORDS = {
    "кім", "кімнің", "кімге", "кімді", "кімде", "кімнен",
    "не", "ненің", "неге", "нені", "неде", "неден",
    "қайда", "қайдан", "қайдасың", "қайдасыз",
    "қашан", "қалай", "неше", "қанша", "неліктен",
    "қай", "қайсы", "қайсысы",
}

def apply_kazakh_rules(words: list, model_labels: list) -> list:
    """
    Post-process model predictions with hard Kazakh grammar rules.
    Improves QUESTION F1 significantly without hurting PERIOD/COMMA.
    """
    labels = model_labels.copy()
    n = len(words)

    # ── Rule 1: Question particles → force QUESTION ───────────────
    # Scan for question particles. They typically appear at end of
    # a question sentence. Force QUESTION on the particle itself.
    for i, word in enumerate(words):
        if word in QUESTION_PARTICLES:
            labels[i] = "QUESTION"
            # If the token before was marked PERIOD, remove it
            # (the period was the sentence boundary, but particle is last)
            if i > 0 and labels[i-1] == "PERIOD":
                labels[i-1] = "O"

    # ── Rule 2: Last token must have PERIOD or QUESTION ───────────
    # Every row = 2-3 sentences → always ends with sentence boundary
    if n > 0 and labels[n-1] == "O":
        # Check if any question word is in this row
        has_q_word = any(w in QUESTION_WORDS for w in words)
        # Check if a question particle already forced QUESTION
        has_q_particle = any(w in QUESTION_PARTICLES for w in words)
        if has_q_particle:
            labels[n-1] = "QUESTION"
        else:
            labels[n-1] = "PERIOD"

    # ── Rule 3: After QUESTION particle, remove stray PERIOD ──────
    # Prevent double-punctuation: "ма QUESTION барды PERIOD" → fix
    for i in range(1, n):
        if labels[i] == "QUESTION" and labels[i-1] in ("PERIOD", "COMMA"):
            labels[i-1] = "O"

    return labels


# ══════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════

_PUNCT_RE = re.compile(r"[^\w\s]", re.UNICODE)

def strip_and_label(text: str):
    text = text.strip()
    if not text:
        return None, None
    tokens = text.split()
    clean_tokens, labels = [], []
    for tok in tokens:
        if not tok:
            continue
        if tok.endswith("?"):
            label, clean = "QUESTION", tok[:-1]
        elif tok.endswith(("!", ".")):
            label, clean = "PERIOD", tok[:-1]
        elif tok.endswith(","):
            label, clean = "COMMA", tok[:-1]
        else:
            label, clean = "O", tok
        clean = _PUNCT_RE.sub("", clean).lower().strip()
        if clean:
            clean_tokens.append(clean)
            labels.append(label)
    if not clean_tokens:
        return None, None
    return " ".join(clean_tokens), " ".join(labels)


def load_hf_corpus(max_rows: int) -> pd.DataFrame:
    if not HF_AVAILABLE:
        print("  datasets library not available.")
        return pd.DataFrame(columns=["input_text", "labels"])
    print("Loading HuggingFace Kazakh corpus…")
    try:
        ds = load_dataset(
            "kz-transformers/multidomain-kazakh-dataset",
            split="train", streaming=True,
        )
    except Exception as e:
        print(f"  ⚠️  HF load failed: {e}\n  Falling back to gold only.")
        return pd.DataFrame(columns=["input_text", "labels"])

    rows, seen = [], set()
    try:
        for item in ds:
            if len(rows) >= max_rows:
                break
            text = item.get("text") or item.get("content") or item.get("sentence") or ""
            if not text or len(text) < 30:
                continue
            chunks = re.split(r"(?<=[.!?])\s+", text)
            for i in range(0, len(chunks) - 1, 2):
                chunk = " ".join(chunks[i: i+2]).strip()
                if len(chunk) < 20:
                    continue
                inp, lbl = strip_and_label(chunk)
                if inp is None:
                    continue
                toks, lbls = inp.split(), lbl.split()
                if len(toks) != len(lbls) or not (5 <= len(toks) <= 60):
                    continue
                key = inp[:60]
                if key in seen:
                    continue
                seen.add(key)
                rows.append({"input_text": inp, "labels": lbl})
    except Exception as e:
        print(f"  ⚠️  Stream interrupted: {e}")

    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["input_text", "labels"])
    print(f"  Collected {len(df):,} rows from HF corpus.")
    return df


def build_training_data(cfg: dict) -> pd.DataFrame:
    corpus_df = load_hf_corpus(cfg["max_corpus_rows"])
    gold_df   = pd.read_csv(cfg["train_path"])[["input_text", "labels"]]
    print(f"  Gold examples: {len(gold_df):,}")

    all_df = pd.concat([corpus_df, gold_df], ignore_index=True)
    all_df = all_df.dropna(subset=["input_text", "labels"])

    if len(all_df) < 2000:
        print(f"  ⚠️  Only {len(all_df)} rows — repeating 10x")
        all_df = pd.concat([all_df] * 10, ignore_index=True)

    all_df = all_df.sample(frac=1, random_state=cfg["seed"]).reset_index(drop=True)
    print(f"  Total training rows: {len(all_df):,}")

    all_labels = " ".join(all_df["labels"]).split()
    total = len(all_labels)
    for lbl in ["O", "COMMA", "PERIOD", "QUESTION"]:
        cnt = all_labels.count(lbl)
        print(f"    {lbl:10s}: {cnt:7,}  ({100*cnt/total:.1f}%)")
    return all_df


# ══════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════

class PunctDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        words       = row["input_text"].split()
        word_labels = row["labels"].split() if "labels" in row and pd.notna(row.get("labels")) else ["O"]*len(words)

        enc = self.tokenizer(
            words, is_split_into_words=True,
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        word_ids  = enc.word_ids(batch_index=0)
        label_ids = []
        prev_wid  = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_wid:
                lbl = word_labels[wid] if wid < len(word_labels) else "O"
                label_ids.append(LABEL2ID[lbl])
            else:
                label_ids.append(-100)
            prev_wid = wid

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


# ══════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════

def build_model(cfg):
    model = AutoModelForTokenClassification.from_pretrained(
        cfg["model_name"], num_labels=NUM_LABELS,
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    return model.to(DEVICE)


def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    for i, batch in enumerate(loader):
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        lbls  = batch["labels"].to(DEVICE)

        with autocast(enabled=USE_AMP):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = loss_fn(logits.view(-1, NUM_LABELS), lbls.view(-1))

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        # Print progress every 100 batches so you know it's alive
        if (i + 1) % 100 == 0:
            print(f"    batch {i+1}/{len(loader)}  loss={total_loss/(i+1):.4f}", flush=True)

    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    all_true, all_pred = [], []
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        lbls  = batch["labels"]
        with autocast(enabled=USE_AMP):
            preds = model(input_ids=ids, attention_mask=mask).logits.argmax(-1).cpu()
        m = lbls != -100
        all_true.extend(lbls[m].tolist())
        all_pred.extend(preds[m].tolist())
    target_ids = [LABEL2ID[l] for l in PUNCT_LABELS]
    return f1_score(all_true, all_pred, labels=target_ids, average="macro", zero_division=0)


def train_model(train_df, val_df, cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
    model     = build_model(cfg)

    weights  = [cfg["class_weights"][ID2LABEL[i]] for i in range(NUM_LABELS)]
    loss_fn  = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float).to(DEVICE),
        ignore_index=-100,
    )

    train_ds = PunctDataset(train_df, tokenizer, cfg["max_len"])
    val_ds   = PunctDataset(val_df,   tokenizer, cfg["max_len"])

    # ── num_workers=0 is REQUIRED on Kaggle — any other value deadlocks ──
    train_ld = DataLoader(train_ds, batch_size=cfg["batch_size"],
                          shuffle=True,  num_workers=0, pin_memory=False)
    val_ld   = DataLoader(val_ds,   batch_size=cfg["batch_size"]*2,
                          shuffle=False, num_workers=0, pin_memory=False)

    optimizer   = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps = len(train_ld) * cfg["epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * cfg["warmup_ratio"]), total_steps
    )
    scaler    = GradScaler(enabled=USE_AMP)
    best_f1   = 0.0
    best_path = "/kaggle/working/best_model.pt"

    for epoch in range(1, cfg["epochs"] + 1):
        print(f"\n  ── Epoch {epoch}/{cfg['epochs']} ──", flush=True)
        train_loss = train_epoch(model, train_ld, optimizer, scheduler, loss_fn, scaler)
        val_f1     = eval_epoch(model, val_ld)
        print(f"  loss={train_loss:.4f}  val_macro_F1={val_f1:.4f}", flush=True)
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Best saved  (F1={best_f1:.4f})", flush=True)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"\nBest val macro F1: {best_f1:.4f}")
    return model, tokenizer


# ══════════════════════════════════════════════
# INFERENCE  (with Kazakh rule post-processing)
# ══════════════════════════════════════════════

@torch.no_grad()
def predict(model, tokenizer, test_df, cfg):
    model.eval()
    all_preds = []

    for _, row in test_df.iterrows():
        words        = row["input_text"].split()
        expected_len = len(words)

        enc = tokenizer(
            words, is_split_into_words=True,
            max_length=512, padding=False,
            truncation=True, return_tensors="pt",
        )
        ids    = enc["input_ids"].to(DEVICE)
        mask   = enc["attention_mask"].to(DEVICE)
        with autocast(enabled=USE_AMP):
            logits = model(input_ids=ids, attention_mask=mask).logits[0]

        word_ids   = enc.word_ids(batch_index=0)
        word_preds = {}
        for tok_i, wid in enumerate(word_ids):
            if wid is not None and wid not in word_preds:
                word_preds[wid] = ID2LABEL[logits[tok_i].argmax().item()]

        # Raw model predictions
        raw_labels = [word_preds.get(i, "O") for i in range(expected_len)]

        # ── Apply Kazakh grammar rules ────────────────────────────
        final_labels = apply_kazakh_rules(words, raw_labels)

        all_preds.append(" ".join(final_labels))

    out = test_df[["id"]].copy()
    out["labels"] = all_preds
    return out


# ══════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════

def validate_submission(test_df, sub_df):
    errors = []
    for _, row in test_df.iterrows():
        exp = len(row["input_text"].split())
        sub = sub_df.loc[sub_df["id"] == row["id"], "labels"]
        if sub.empty:
            errors.append(f"MISSING {row['id']}")
            continue
        got = len(sub.values[0].split())
        if exp != got:
            errors.append(f"LENGTH MISMATCH {row['id']}: exp={exp} got={got}")
    if errors:
        print(f"⚠️  {len(errors)} errors:")
        for e in errors[:5]: print(f"   {e}")
    else:
        print(f"✅  Submission valid — {len(sub_df):,} rows, all lengths match.")


# ══════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════

def main():
    import time
    t0 = time.time()

    print("=" * 60)
    print("KazPunct — Fast + High Score Solution")
    print("=" * 60)

    test_df  = pd.read_csv(CFG["test_path"])
    print(f"Test set: {len(test_df):,} rows")

    train_df = build_training_data(CFG)

    val_size  = max(500, int(len(train_df) * 0.10))
    val_df    = train_df.iloc[:val_size].copy()
    train_df2 = train_df.iloc[val_size:].copy()
    print(f"Train: {len(train_df2):,}  Val: {len(val_df):,}")

    model, tokenizer = train_model(train_df2, val_df, CFG)

    print("\nGenerating predictions (with Kazakh grammar rules)…")
    submission = predict(model, tokenizer, test_df, CFG)

    validate_submission(test_df, submission)

    submission.to_csv(CFG["output_path"], index=False)
    elapsed = (time.time() - t0) / 60
    print(f"\n✅  Done in {elapsed:.1f} minutes")
    print(f"Submission saved → {CFG['output_path']}")
    print(submission.head(5).to_string())


if __name__ == "__main__":
    main()

Device: cuda  |  Mixed precision (fp16): True
KazPunct — Fast + High Score Solution
Test set: 3,552 rows
Loading HuggingFace Kazakh corpus…


README.md: 0.00B [00:00, ?B/s]

  Collected 40,002 rows from HF corpus.
  Gold examples: 500
  Total training rows: 40,502
    O         : 654,682  (84.2%)
    COMMA     :  49,714  (6.4%)
    PERIOD    :  70,782  (9.1%)
    QUESTION  :   2,299  (0.3%)
Train: 36,452  Val: 4,050


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



  ── Epoch 1/14 ──
    batch 100/1140  loss=1.3261
    batch 200/1140  loss=1.1433
    batch 300/1140  loss=0.9522
    batch 400/1140  loss=0.8275
    batch 500/1140  loss=0.7366
    batch 600/1140  loss=0.6715
    batch 700/1140  loss=0.6222
    batch 800/1140  loss=0.5837
    batch 900/1140  loss=0.5542
    batch 1000/1140  loss=0.5298
    batch 1100/1140  loss=0.5088
  loss=0.5019  val_macro_F1=0.6109
  ✓ Best saved  (F1=0.6109)

  ── Epoch 2/14 ──
    batch 100/1140  loss=0.2445
    batch 200/1140  loss=0.2462
    batch 300/1140  loss=0.2492
    batch 400/1140  loss=0.2477
    batch 500/1140  loss=0.2448
    batch 600/1140  loss=0.2420
    batch 700/1140  loss=0.2402
    batch 800/1140  loss=0.2385
    batch 900/1140  loss=0.2364
    batch 1000/1140  loss=0.2365
    batch 1100/1140  loss=0.2355
  loss=0.2354  val_macro_F1=0.7290
  ✓ Best saved  (F1=0.7290)

  ── Epoch 3/14 ──
    batch 100/1140  loss=0.1626
    batch 200/1140  loss=0.1685
    batch 300/1140  loss=0.1712
    batch 